# ERI integral evaluation: some tests
This notebook will serve as a brief playground to compare and test the three integral codes: PySCF, Oatmeal and Interest, in order to define an appropriate wrapper for Interest (and maybe Oatmeal at some point), to obtain the full ERI-ao tensor. As always, we will start with the imports:

In [1]:
from typing import List

import numpy as np
from numpy.typing import NDArray
from numpy.ctypeslib import ndpointer

import ctypes as ct

from pyscf import gto

from py_mods.src.integrals.GTO import GTO
from py_mods.src.integrals.CGTO import create_CGTOClass, Eri_GTO_tensor


---

# Same angular momenta shells
## PySCF: the reference
We will be using PySCF as a reference, as all the internal code can yield the ERI-AO tensor directly, while the other codes at this point only yield shell-based ERI-AO tensors. We will start by building the ERI-AO references of same angular momenta $s$, $p$ or $d$ basis composed of a single primitive:

In [2]:
l_tags = ["S", "P", "D", "F", "G", "H"]
reference_eri_tensors = []

for l in range(0, 4):
    mol = gto.M(
        atom="H 0 0 0; H 0 0 1.4;",
        unit="Bohr",
        basis={
            "H": gto.basis.parse(
                f"""
            # This is the standard STO-3G definition
            # Exponent    Contraction Coeff
            H {l_tags[l]}
            3.0    0.15
            # 1    1
        """  # recall that is exponent, coefficient
            )
        },
        cart=True,
    )

    mol.build()

    overlap = mol.intor("int1e_ovlp")
    kin = mol.intor("int1e_kin")
    V = mol.intor("int1e_nuc")
    ref_eri = mol.intor("int2e")

    norm_vec = 1.0 / np.sqrt(np.diag(overlap))

    ref_eri *= norm_vec[:, None, None, None]
    ref_eri *= norm_vec[None, :, None, None]
    ref_eri *= norm_vec[None, None, :, None]
    ref_eri *= norm_vec[None, None, None, :]

    reference_eri_tensors.append(np.array(ref_eri))

eri_s, eri_p, eri_d,eri_f = reference_eri_tensors

print(f"S tensor shape is: {eri_s.shape}")
print(f"P tensor shape is: {eri_p.shape}")
print(f"D tensor shape is: {eri_d.shape}")
print(f"F tensor shape is: {eri_f.shape}")

S tensor shape is: (2, 2, 2, 2)
P tensor shape is: (6, 6, 6, 6)
D tensor shape is: (12, 12, 12, 12)
F tensor shape is: (20, 20, 20, 20)


And we will define some mixed shells, due to the eventual need of mixed shells 

In [3]:
l_tags = ["S", "P", "D", "F", "G", "H"]
reference_eri_tensors = []

for l in range(0,3):
    mol = gto.M(
        atom="H 0 0 0; H 0 0 1.4;",
        unit="Bohr",
        basis={
            "H": gto.basis.parse( 
                f"""
            # This is the standard STO-3G definition
            # Exponent    Contraction Coeff
            H {l_tags[l]}
            3.0    0.15
            H {l_tags[l+1]}
            3.0    0.15
        """ #recall that is exponent, coefficient
            )
        },
        cart=True,
    )

    

    mol.build()


    overlap = mol.intor("int1e_ovlp")
    kin = mol.intor("int1e_kin")
    V = mol.intor("int1e_nuc")
    ref_eri = mol.intor("int2e")

    norm_vec = 1.0 / np.sqrt(np.diag(overlap))

    ref_eri *= norm_vec[:, None, None, None]
    ref_eri *= norm_vec[None, :, None, None]
    ref_eri *= norm_vec[None, None, :, None]
    ref_eri *= norm_vec[None, None, None, :]

    reference_eri_tensors.append(np.array(ref_eri))

eri_sp, eri_pd, eri_df = reference_eri_tensors

print(f'SP tensor shape is: {eri_sp.shape}')
print(f'PD tensor shape is: {eri_pd.shape}')
print(f'DF tensor shape is: {eri_df.shape}')


SP tensor shape is: (8, 8, 8, 8)
PD tensor shape is: (18, 18, 18, 18)
DF tensor shape is: (32, 32, 32, 32)



---

## Oatmeal
Now we will try to obtain the same results with the Oatmeal code, but we need a wrapper!! Because at this point we can only obtain the shell tensors:


In [4]:
def naive_wrapper(gto_list: List[GTO]) -> NDArray[np.float64]: 
    projections = [len(primitive.l_projections) for primitive in gto_list]
    shell_start = [sum(projections[0:i]) for i in range(len(projections))]
    tensor_size = (sum(projections))

    # print(f"Number of projections are: {tensor_size}")
    # print(f"Projections per shell are: {projections}")
    # print(shell_start)

    eri_tensor = np.zeros([tensor_size for i in range(4)])
    # print(f'Tensor size is: {eri_tensor.shape}')

    for P, p_gto in enumerate(gto_list):
        p_size = projections[P]
        p_start = shell_start[P]
        p_end = shell_start[P] + p_size
        for Q, q_gto in enumerate(gto_list):
            q_size = projections[Q]
            q_start = shell_start[Q]
            q_end = shell_start[Q] + q_size
            for R, r_gto in enumerate(gto_list):
                r_size = projections[R]
                r_start = shell_start[R]
                r_end = shell_start[R] + r_size
                for S, s_gto in enumerate(gto_list):
                    s_size = projections[S]
                    s_start = shell_start[S]
                    s_end = shell_start[S] + s_size
                    eri_tensor[p_start:p_end, q_start:q_end, r_start:r_end, s_start:s_end] = Eri_GTO_tensor(p_gto, q_gto, r_gto, s_gto)

    return eri_tensor

In [5]:
def single_shell_spd_test(ref_eri_list: List[NDArray]):
    oatmeal_eris = []
    eri_s, eri_p, eri_d = ref_eri_list

    for l in range(0, 3):
        r1 = np.array([0.0, 0.0, 0.0])
        r2 = np.array([0.0, 0.0, 1.4])
        H_exps = np.array([3])
        H_coeffs = np.array([0.15])

        # Construct the CGTO
        atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l)
        atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l)

        gto_list = [atom_1_basis, atom_2_basis]

        Eri_test: NDArray[np.float64] = naive_wrapper(gto_list)

        oatmeal_eris.append((Eri_test))

    oat_eri_s, oat_eri_p, oat_eri_d = oatmeal_eris

    print(f"S tensor is equivalent to pyscf's: {np.allclose(oat_eri_s, eri_s)}")
    print(f"P tensor is equivalent to pyscf's: {np.allclose(oat_eri_p, eri_p)}")
    print(f"D tensor is equivalent to pyscf's: {np.allclose(oat_eri_d, eri_d)}")

    return (
        np.allclose(oat_eri_s, eri_s)
        and np.allclose(oat_eri_p, eri_p)
        and np.allclose(oat_eri_d, eri_d)
    )

In [6]:
def mixed_shell_sp_test(eri_sp):
    oatmeal_eris = []

    for l in range(0, 1):
        r1 = np.array([0.0, 0.0, 0.0])
        r2 = np.array([0.0, 0.0, 1.4])
        H_exps = np.array([3])
        H_coeffs = np.array([0.15])

        # Construct the CGTO
        atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l)
        atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l)
        atom_1_basis_2 = create_CGTOClass(r1, H_exps, H_coeffs, l+1)
        atom_2_basis_2 = create_CGTOClass(r2, H_exps, H_coeffs, l+1)

        gto_list = [atom_1_basis, atom_1_basis_2, atom_2_basis, atom_2_basis_2]

        Eri_test: NDArray[np.float64] = naive_wrapper(gto_list)

        oatmeal_eris.append((Eri_test))

    oat_eri_s = oatmeal_eris

    print(f"S tensor is equivalent to pyscf's: {np.allclose(oat_eri_s, eri_sp)}")

    return np.allclose(oat_eri_s, eri_sp)

In [7]:
print(f'\n\nSingle shell for S, P and D passed: {single_shell_spd_test([eri_s, eri_p, eri_d])}')
print(f'\n\nMixed shell for SP passed: {mixed_shell_sp_test(eri_sp)}')

S tensor is equivalent to pyscf's: True
P tensor is equivalent to pyscf's: True
D tensor is equivalent to pyscf's: True


Single shell for S, P and D passed: True
S tensor is equivalent to pyscf's: True


Mixed shell for SP passed: True


Which is cool, but takes $\approx 50$ seconds for just two $d$ primitives. Therefore this is why we need to play with Interest. 

---

## Interest


The Interest interface to C is defined as:

```Fortran
subroutine interest_prim_cls_interface( &
                factor, fint_inp, &
                la_inp, alpha_inp, ax_inp, ay_inp, az_inp, anorm_inp, &
                lb_inp, beta_inp, bx_inp, by_inp, bz_inp, bnorm_inp, &
                lc_inp, gamma_inp, cx_inp, cy_inp, cz_inp, cnorm_inp, &
                ld_inp, delta_inp, dx_inp, dy_inp, dz_inp, dnorm_inp) &
    bind(C, name="interest_prim_py")

    use iso_c_binding, only: c_int, c_double
    use module_interest_eri
    implicit none

    character(len=4) :: class
    real(c_double), value, intent(in) :: factor
    real(c_double), intent(inout) :: fint_inp(1000)

    integer(c_int), value, intent(in) :: la_inp, lb_inp, lc_inp, ld_inp
    real(c_double), value, intent(in) :: alpha_inp, ax_inp, ay_inp, az_inp, anorm_inp
    real(c_double), value, intent(in) :: beta_inp,  bx_inp, by_inp, bz_inp, bnorm_inp
    real(c_double), value, intent(in) :: gamma_inp, cx_inp, cy_inp, cz_inp, cnorm_inp
    real(c_double), value, intent(in) :: delta_inp, dx_inp, dy_inp, dz_inp, dnorm_inp


    call interest_initialize( .false. ) 

    class = 'llll'
    fint_inp = 0.0d0

    call interest_eri(class,factor,fint_inp, &
        la_inp,alpha_inp,ax_inp,ay_inp,az_inp,anorm_inp, &
        lb_inp,beta_inp, bx_inp,by_inp,bz_inp,bnorm_inp, &
        lc_inp,gamma_inp,cx_inp,cy_inp,cz_inp,cnorm_inp, &
        ld_inp,delta_inp,dx_inp,dy_inp,dz_inp,dnorm_inp)

end subroutine interest_prim_cls_interface
```

And once compiled can be called as:

In [8]:
lib = ct.CDLL("/Users/fernando.mata/bin/Oatmeal/notebooks/interest_interface/interest/src/inter_py_lib.so")

lib.interest_prim_py.argtypes = [
    ct.c_double,                                                # factor
    ndpointer(dtype=np.float64, ndim=1, shape=(10000,)),         # fint_inp

    ct.c_int,                                                   # la_inp
    ct.c_double, ct.c_double, ct.c_double, ct.c_double, ct.c_double,

    ct.c_int,                                                   # lb_inp
    ct.c_double, ct.c_double, ct.c_double, ct.c_double, ct.c_double,

    ct.c_int,                                                   # lc_inp
    ct.c_double, ct.c_double, ct.c_double, ct.c_double, ct.c_double,

    ct.c_int,                                                   # ld_inp
    ct.c_double, ct.c_double, ct.c_double, ct.c_double, ct.c_double,
]
lib.interest_prim_py.restype = None

fint = np.zeros(10000, dtype=np.float64)
Na_buffer = np.zeros([10000])

def interest_from_gto(
    gto1: GTO, gto2: GTO, gto3: GTO, gto4: GTO, working_array
) -> None:
    fact =  1

    la = gto1.total_L + 1
    lb = gto2.total_L + 1
    lc = gto3.total_L + 1
    ld = gto4.total_L + 1

    ax, ay, az = gto1.R
    bx, by, bz = gto2.R
    cx, cy, cz = gto3.R
    dx, dy, dz = gto4.R

    a_exp = gto1.exp 
    b_exp = gto2.exp 
    c_exp = gto3.exp 
    d_exp = gto4.exp 

    lib.interest_prim_py(
        fact, working_array,
        la, a_exp, ax, ay, az, 1,
        lb, b_exp, bx, by, bz, 1,
        lc, c_exp, cx, cy, cz, 1,
        ld, d_exp, dx, dy, dz, 1,
    )

In [9]:
def naive_wrapper_interest(gto_list: List[GTO], buffer_1) -> NDArray[np.float64]: 
    projections = [len(primitive.l_projections) for primitive in gto_list]
    shell_start = [sum(projections[0:i]) for i in range(len(projections))]
    tensor_size = (sum(projections))

    # print(f"Number of projections are: {tensor_size}")
    # print(f"Projections per shell are: {projections}")
    # print(shell_start)

    eri_tensor = np.zeros([tensor_size for _ in range(4)])
    # print(f'Tensor size is: {eri_tensor.shape}')

    for P, p_gto in enumerate(gto_list):
        p_size = projections[P]
        p_start = shell_start[P]
        p_end = shell_start[P] + p_size
        p_consts = p_gto.normalization_constants

        for Q, q_gto in enumerate(gto_list):
            q_size = projections[Q]
            q_start = shell_start[Q]
            q_end = shell_start[Q] + q_size
            q_consts = q_gto.normalization_constants

            for R, r_gto in enumerate(gto_list):
                r_size = projections[R]
                r_start = shell_start[R]
                r_end = shell_start[R] + r_size
                r_consts = r_gto.normalization_constants

                for S, s_gto in enumerate(gto_list):
                    s_size = projections[S]
                    s_start = shell_start[S]
                    s_end = shell_start[S] + s_size
                    s_consts = s_gto.normalization_constants

                    shell_total_size = p_size * q_size * r_size * s_size

                    buffer_1[:shell_total_size] = 0.0
                    interest_from_gto(p_gto, q_gto, r_gto, s_gto, buffer_1)

                    # if P != Q and R == S:
                    #     block = buffer_1[:shell_total_size].reshape((q_size, p_size, r_size, s_size)).transpose(1,0,2,3)

                    # elif P == Q and R != S:
                    #     block = buffer_1[:shell_total_size].reshape((p_size, q_size, s_size, r_size)).transpose(0,1,3,2)

                    # else:
                    block = buffer_1[:shell_total_size].reshape((p_size, q_size, r_size, s_size))

                    norm_tensor = np.einsum(
                        "i,j,k,l->ijkl", p_consts, q_consts, r_consts, s_consts
                    )

                    block *= norm_tensor

                    assert block.shape == (p_size, q_size, r_size, s_size)

                    eri_tensor[p_start:p_end, q_start:q_end, r_start:r_end, s_start:s_end] = block

    return eri_tensor

In [10]:
def interest_single_shell_spd_test(ref_eri_list: List[NDArray], buffer_1):
    oatmeal_eris = []
    eri_s, eri_p, eri_d, eri_f = ref_eri_list

    for l in range(0, 4):
        r1 = np.array([0.0, 0.0, 0.0])
        r2 = np.array([0.0, 0.0, 1.4])
        H_exps = np.array([3])
        H_coeffs = np.array([0.15])

        # Construct the CGTO
        atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l).primitives[0]
        atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l).primitives[0]

        gto_list = [atom_1_basis, atom_2_basis]

        Eri_test: NDArray[np.float64] = naive_wrapper_interest(gto_list, buffer_1)

        oatmeal_eris.append((Eri_test))

    oat_eri_s, oat_eri_p, oat_eri_d, oat_eri_f = oatmeal_eris

    print(f"S tensor is equivalent to pyscf's: {np.allclose(oat_eri_s, eri_s)}")
    print(f"P tensor is equivalent to pyscf's: {np.allclose(oat_eri_p, eri_p)}")
    print(f"D tensor is equivalent to pyscf's: {np.allclose(oat_eri_d, eri_d)}")
    print(f"F tensor is equivalent to pyscf's: {np.allclose(oat_eri_f, eri_f)}")


    return (
        np.allclose(oat_eri_s, eri_s)
        and np.allclose(oat_eri_p, eri_p)
        and np.allclose(oat_eri_d, eri_d)
    )

In [11]:
interest_single_shell_spd_test([eri_s, eri_p, eri_d, eri_f], fint)

S tensor is equivalent to pyscf's: True
P tensor is equivalent to pyscf's: False
D tensor is equivalent to pyscf's: False
F tensor is equivalent to pyscf's: False


False

In [12]:
def mixed_shell_sp_test(eri_sp, buffer_1):
    oatmeal_eris = []
    l=0
    
    r1 = np.array([0.0, 0.0, 0.0])
    r2 = np.array([0.0, 0.0, 1.4])
    H_exps = np.array([3])
    H_coeffs = np.array([0.15])

    # Construct the CGTO
    atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l).primitives[0]
    atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l).primitives[0]
    atom_1_basis_2 = create_CGTOClass(r1, H_exps, H_coeffs, l+1).primitives[0]
    atom_2_basis_2 = create_CGTOClass(r2, H_exps, H_coeffs, l+1).primitives[0]

    gto_list = [atom_1_basis, atom_2_basis, atom_1_basis_2, atom_2_basis_2]

    Eri_test: NDArray[np.float64] = naive_wrapper_interest(gto_list, buffer_1)

    oatmeal_eris.append((Eri_test))

    oat_eri_s = oatmeal_eris

    print(f"SP tensor is equivalent to pyscf's: {np.allclose(oat_eri_s, eri_sp)}")

    return np.allclose(oat_eri_s, eri_sp)

In [13]:
mixed_shell_sp_test(eri_sp, fint)

SP tensor is equivalent to pyscf's: False


False

In [14]:
eri_l = [eri_s, eri_p, eri_d]

r1 = np.array([0.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.4])
H_exps = np.array([1])
H_coeffs = np.array([1])

l = 1
cart_l = int((l + 1) * (l + 2) / 2)

atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l).primitives[0]
atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l).primitives[0]
# atom_1_basis_2 = create_CGTOClass(r1, H_exps, H_coeffs, l).primitives[0]
# atom_2_basis_2 = create_CGTOClass(r2, H_exps, H_coeffs, l).primitives[0]

print(atom_1_basis.total_L)
print(atom_1_basis.normalization_constants)

test_integral = interest_from_gto(
    atom_1_basis, atom_1_basis, atom_1_basis, atom_1_basis, fint
)

na_tensor = np.einsum(
    "i,j,k,l->ijkl",
    atom_1_basis.normalization_constants,
    atom_1_basis.normalization_constants,
    atom_1_basis.normalization_constants,
    atom_1_basis.normalization_constants,
).reshape(-1)

Na_buffer[0 : len(na_tensor)] = na_tensor

fint *= Na_buffer

reshaped = fint[0 : cart_l**4].reshape([cart_l, cart_l, cart_l, cart_l])
print(reshaped.shape)
print(np.allclose(reshaped, eri_l[l][0:cart_l,0:cart_l,0:cart_l,0:cart_l]))

1
[1.42541094 1.42541094 1.42541094]
(3, 3, 3, 3)
False


In [15]:
oatmeal_eris = []
l=0

r1 = np.array([0.0, 0.0, 0.0])
r2 = np.array([0.0, 0.0, 1.4])
H_exps = np.array([3])
H_coeffs = np.array([0.15])

# Construct the CGTO
atom_1_basis = create_CGTOClass(r1, H_exps, H_coeffs, l).primitives[0]
atom_2_basis = create_CGTOClass(r2, H_exps, H_coeffs, l).primitives[0]
atom_1_basis_2 = create_CGTOClass(r1, H_exps, H_coeffs, l+1).primitives[0]
atom_2_basis_2 = create_CGTOClass(r2, H_exps, H_coeffs, l+1).primitives[0]

gto_list = [atom_1_basis, atom_2_basis, atom_1_basis_2, atom_2_basis_2]

Eri_test: NDArray[np.float64] = naive_wrapper_interest(gto_list, fint)

oatmeal_eris.append((Eri_test))

oat_eri_s = oatmeal_eris[0]

print(f"SP tensor is equivalent to pyscf's: {np.allclose(oat_eri_s, eri_sp)}")
print(oat_eri_s.shape)
print(eri_sp.shape)

SP tensor is equivalent to pyscf's: False
(8, 8, 8, 8)
(8, 8, 8, 8)


In [16]:
print(oat_eri_s[:,:,0,0])
print(eri_sp[:,:,0,0])
print(oat_eri_s[:,:,0,0] - eri_sp.transpose(2,3,1,0)[:,:,0,0])

[[ 1.95441005  0.06899652  0.          0.          0.          0.
   0.         -0.18596446]
 [ 0.06899652  0.71385345  0.          0.          0.14865087  0.
   0.         -0.14606782]
 [ 0.          0.          1.62867504  0.          0.          0.06130259
   0.          0.        ]
 [ 0.          0.          0.          1.62867504  0.          0.
   0.06130259  0.        ]
 [ 0.          0.14865087  0.          0.          1.62867504  0.
   0.         -0.33319325]
 [ 0.          0.          0.06130259  0.         -0.          0.68373477
   0.         -0.        ]
 [ 0.          0.          0.          0.06130259 -0.          0.
   0.68373477  0.        ]
 [-0.18596446 -0.14606782  0.          0.         -0.33319325  0.
   0.          0.77135972]]
[[ 1.95441005  0.          0.          0.          0.06899652  0.
   0.         -0.18596446]
 [ 0.          1.62867504  0.          0.          0.          0.06130259
   0.         -0.        ]
 [ 0.          0.          1.62867504  0.    